# 02_Feature Engineering - Extracción de Características

**Objetivo**: Transformar las series temporales de los sensores en un conjunto de características estadísticas que resuman su comportamiento. Este dataset tabular será la base para el modelo de clasificación de accidentes.

**Contexto**: En el notebook anterior (`NPPAD_01_EDA`) vimos que 5 sensores eran insuficientes para cubrir todos los tipos de accidentes. Por ello, ampliamos el conjunto a **12 sensores** que cubren las principales áreas del reactor:

| Área | Sensores | Variables |
|------|----------|-----------|
| **Primario** | P, TAVG, LVPZ, WBK | Presión, temperatura media, nivel del presurizador, flujo de rotura |
| **Secundario** | LSGA, LSGB, PSGA, WSTA | Niveles de SG, presión de SG (A), flujo de vapor (A)|
| **Neutrónica** | QMWT, PWNT | Potencia térmica, flujo neutrónico |
| **Contención** | LWRB, TPCT | Nivel de agua en colector, temperatura pico de vaina |

Esta selección se basa en la literatura técnica (NUREG/CR-xxxx) y en el conocimiento de la física del reactor, asegurando que se capturan los comportamientos característicos de los accidentes más relevantes (LOCA, ATWS, LOF, LACP, SGBTR, etc.).

---

## 1. Configuración inicial

Importamos las librerías necesarias y configuramos el proyecto. Definimos las rutas de datos de entrada (los CSVs de operación) y de salida (el dataset de características).

In [1]:
import sys
import os
import pandas as pd

# Añadir la raíz del proyecto al path para importar src
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.features import procesar_todos_archivos, COLUMNAS_12, sensores
from src.utils import crear_carpeta

# Rutas
DATA_PATH = "../data/processed/operation/"
OUTPUT_CSV = "../data/processed/features_12sensores_1217muestras.csv"

## 2. Procesamiento de archivos
La función `procesar_todos_archivos` recorre todas las carpetas (una por tipo de accidente) y todos los archivos CSV dentro de cada carpeta. Cada archivo corresponde a una severidad diferente del mismo accidente (ej. `1.csv` = severidad base, etc.).

De cada archivo se extrae:

`accidente`: nombre de la carpeta (ej. `LOCA`, `ATWS`).

`severidad`: número entero extraído del nombre del archivo (ej. `1`, `2`, ...). En caso de archivos con guion inicial (`-1.csv`), se limpia automáticamente.

In [2]:
print("Procesando archivos...")
dict_list = procesar_todos_archivos(DATA_PATH, COLUMNAS_12, sensores)
print(f"Hemos generado {len(dict_list)} diccionarios")

Procesando archivos...
Hemos generado 1217 diccionarios


## 3. Extracción de estadísticas por sensor
Para cada sensor y cada archivo, se calculan las siguientes estadísticas:

| Estadística | Descripción | Relevancia |
|------|----------|-----------|
| **Media** | Valor promedio del sensor durante la simulación | Indica el nivel general de la variable |
| **Desviación estándar** | Variabilidad de la señal | Sensores con alta std son más informativos |
| **Máximo** | Valor pico alcanzado | Picos indican eventos extremos (ej. flujo de rotura) |
| **Mínimo** | Valor mínimo | Caídas bruscas son típicas de roturas |
| **Valor inicial** | Primer valor de la serie | Estado inicial del accidente |
| **Valor final** | Último valor | Estado final o estabilizado |
| **Pendiente de regresión** | Tasa de cambio global (lineal) | Tendencia general (creciente/decreciente) |
| **Tiempo hasta el máximo** | Instante en que se alcanza el pico | Cuándo ocurre el evento más extremo |
| **Tiempo hasta el mínimo** | Instante en que se alcanza el valle | Cuándo ocurre la caída más profunda |

Estas estadísticas capturan tanto los **valores absolutos** como la **dinámica temporal** de cada sensor, lo que resultará clave para el clasificador (la extracción se realiza dentro de procesar_todos_archivos usando la función extraer_estadisticas (definida en `src/features.py`)

## 4. Construcción del dataset final
Los diccionarios generados se convierten en un DataFrame de pandas. Cada fila corresponde a un archivo (una simulación de un accidente con una severidad específica) y cada columna a una estadística de un sensor.

**Dimensiones del dataset**: 1217 filas (simulaciones) × 110 columnas (2 de metadatos + 108 estadísticas para 12 sensores).

In [3]:
df_stats = pd.DataFrame(dict_list)
print(f"Nuestro dataframe tiene dimensiones {df_stats.shape}")

Nuestro dataframe tiene dimensiones (1217, 110)


Guardamos el dataset en un archivo CSV para poder reutilizarlo en los siguientes notebooks sin tener que repetir el procesamiento.

In [4]:
crear_carpeta(os.path.dirname(OUTPUT_CSV))
df_stats.to_csv(OUTPUT_CSV, index=False, na_rep='nan')
print(f"Dataframe guardado en {OUTPUT_CSV}")

Dataframe guardado en ../data/processed/features_12sensores_1217muestras.csv


## 5. Conclusión
Hemos transformado 1217 series temporales de 12 sensores en un dataset tabular de 110 características. Este dataset será la entrada para el modelo de clasificación en el siguiente notebook (`NPPAD_03_Modelado_Clasificacion`), donde seleccionaremos las características más relevantes y entrenaremos un Random Forest para diagnosticar el tipo de accidente.